# Smart MCQ Solver — TF-IDF NN Lookup

**Key insight**: test questions are near-duplicates of train questions.  
**Strategy**: TF-IDF (word + char) → cosine similarity → Top-K neighbour voting → option-level matching.  
**CV MAP@3 ≈ 0.997**

In [9]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher
from scipy.sparse import hstack

In [10]:
train = pd.read_csv('/kaggle/input/datasets/kirandwippaul/mcqdata/train.csv')
test = pd.read_csv('/kaggle/input/datasets/kirandwippaul/mcqdata/test.csv')
print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Columns: {train.columns.tolist()}')

Train: (2000, 8) | Test: (500, 7)
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


## 1. Build TF-IDF features (word + char)

In [11]:
def make_text(row):
    return f"{row['prompt']} {row['A']} {row['B']} {row['C']} {row['D']} {row['E']}"

all_texts = pd.concat([train, test], ignore_index=True).apply(make_text, axis=1).tolist()
print(f'Total texts: {len(all_texts)}')

Total texts: 2500


In [12]:
tfidf_word = TfidfVectorizer(ngram_range=(1,2), min_df=1, analyzer='word', sublinear_tf=True)
tfidf_char = TfidfVectorizer(ngram_range=(2,5), min_df=1, analyzer='char_wb', max_features=50000, sublinear_tf=True)

tfidf_word.fit(all_texts)
tfidf_char.fit(all_texts)
print(f'Word features: {len(tfidf_word.vocabulary_)} | Char features: {len(tfidf_char.vocabulary_)}')

Word features: 13798 | Char features: 24842


In [13]:
train_texts = train.apply(make_text, axis=1).tolist()
test_texts = test.apply(make_text, axis=1).tolist()

train_vecs = hstack([tfidf_word.transform(train_texts), tfidf_char.transform(train_texts)])
test_vecs = hstack([tfidf_word.transform(test_texts), tfidf_char.transform(test_texts)])
print(f'Train vec: {train_vecs.shape} | Test vec: {test_vecs.shape}')

Train vec: (2000, 38640) | Test vec: (500, 38640)


## 2. Option-level matching function

In [14]:
K = 5  # number of nearest neighbours

def predict_top3(test_row, sims_row, train_df):
    top_k = np.argsort(sims_row)[-K:][::-1]
    scores = {opt: 0.0 for opt in 'ABCDE'}
    for nn_idx in top_k:
        nn_row = train_df.iloc[nn_idx]
        nn_correct_text = str(nn_row[nn_row['answer']]).strip()
        w = sims_row[nn_idx]
        for opt in 'ABCDE':
            opt_text = str(test_row[opt]).strip()
            if opt_text == nn_correct_text:
                scores[opt] += w
            else:
                scores[opt] += w * SequenceMatcher(None, opt_text, nn_correct_text).ratio()
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [o for o, _ in ranked[:3]]

## 3. Cross-validate (leave-one-out on train)

In [15]:
train_sims = cosine_similarity(train_vecs, train_vecs)
np.fill_diagonal(train_sims, -1)

maps = []
for i in range(len(train)):
    top3 = predict_top3(train.iloc[i], train_sims[i], train)
    correct = train.iloc[i]['answer']
    ap = 1.0 / (top3.index(correct) + 1) if correct in top3 else 0.0
    maps.append(ap)

print(f'CV MAP@3: {np.mean(maps):.6f}')
print(f'Top-1 Acc: {sum(m == 1.0 for m in maps) / len(maps):.4f}')
print(f'In-Top-3: {sum(m > 0 for m in maps) / len(maps):.4f}')

CV MAP@3: 0.996917
Top-1 Acc: 0.9950
In-Top-3: 0.9990


## 4. Generate test predictions

In [16]:
test_train_sims = cosine_similarity(test_vecs, train_vecs)

results = []
for i in range(len(test)):
    top3 = predict_top3(test.iloc[i], test_train_sims[i], train)
    results.append({'ID': test.iloc[i]['id'], 'Prediction': ' '.join(top3)})

submission = pd.DataFrame(results)
print(submission.head(10))

   ID Prediction
0   1      A E B
1   2      B D A
2   3      B E D
3   4      E A D
4   5      C A D
5   6      D E A
6   7      E D C
7   8      B E A
8   9      C D A
9  10      B D A


In [17]:
submission.to_csv('submission.csv', index=False)
print(f'\n✓ Saved: {len(submission)} predictions')
print(f'Sample: {submission.iloc[0].to_dict()}')


✓ Saved: 500 predictions
Sample: {'ID': 1, 'Prediction': 'A E B'}


## 5. Diagnostics

In [18]:
# Check similarity distribution for test queries
best_sims = test_train_sims.max(axis=1)
print(f'Test→Train similarity stats:')
print(f'  Mean: {best_sims.mean():.4f}')
print(f'  Min:  {best_sims.min():.4f}')
print(f'  >0.99: {(best_sims > 0.99).sum()}/500')
print(f'  >0.95: {(best_sims > 0.95).sum()}/500')
print(f'  >0.90: {(best_sims > 0.90).sum()}/500')

Test→Train similarity stats:
  Mean: 0.9930
  Min:  0.8778
  >0.99: 391/500
  >0.95: 490/500
  >0.90: 498/500


In [19]:
# Answer distribution in predictions
from collections import Counter
pred_dist = Counter(r['Prediction'].split()[0] for r in results)
print(f'Predicted answer distribution (top-1):')
for k in sorted(pred_dist): print(f'  {k}: {pred_dist[k]}')
print(f'\nTrain answer distribution:')
print(train['answer'].value_counts().sort_index().to_string())

Predicted answer distribution (top-1):
  A: 87
  B: 114
  C: 119
  D: 96
  E: 84

Train answer distribution:
answer
A    369
B    490
C    459
D    358
E    324
